# Email Analysis

In [1]:
import sys
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

_SRC = Path("..").resolve() / "src"
if str(_SRC) not in sys.path:
    sys.path.insert(0, str(_SRC))

# talon 1.4.4 is incompatible with modern sklearn out of the box
# (sklearn.externals.joblib + pickled sklearn.svm.classes).
from email_cls_with_clustering.talon_compat import init_talon

signature, quotations = init_talon()
print("talon ready")

talon ready


In [2]:
import sys
from pathlib import Path

_SRC = Path("..").resolve() / "src"
if str(_SRC) not in sys.path:
    sys.path.insert(0, str(_SRC))

from email_cls_with_clustering.tracking import setup_tracking

# Local SQLite store: ../mlflow.db. Pass enable_sklearn_autolog=True when training.
experiment = setup_tracking()
print(f"tracking uri ready → experiment={experiment.name} (id={experiment.experiment_id})")


tracking uri ready → experiment=email-cls-with-clustering (id=1)


In [3]:
! kaggle datasets download wcukierski/enron-email-dataset

Dataset URL: https://www.kaggle.com/datasets/wcukierski/enron-email-dataset
License(s): copyright-authors
enron-email-dataset.zip: Skipping, found more recently modified local copy (use --force to force download)


## Preprocess Dataset

In [4]:
# load the dataset
# import zipfile

# with zipfile.ZipFile('enron-email-dataset.zip', 'r') as zip_ref:
#     zip_ref.extractall()

df = pd.read_csv('emails.csv')
df.head()

,file,message
0,allen-p/_sent_mail/1.,Message-ID: <18782981.1075855378110.JavaMail.e...
1,allen-p/_sent_mail/10.,Message-ID: <15464986.1075855378456.JavaMail.e...
2,allen-p/_sent_mail/100.,Message-ID: <24216240.1075855687451.JavaMail.e...
3,allen-p/_sent_mail/1000.,Message-ID: <13505866.1075863688222.JavaMail.e...
4,allen-p/_sent_mail/1001.,Message-ID: <30922949.1075863688243.JavaMail.e...


In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 517401 entries, 0 to 517400
Data columns (total 2 columns):
 #   Column   Non-Null Count   Dtype
---  ------   --------------   -----
 0   file     517401 non-null  str  
 1   message  517401 non-null  str  
dtypes: str(2)
memory usage: 1.3 GB


In [ ]:
# from email_cls_with_clustering.preprocess import expand_emails

# df_expanded = expand_emails(df)

# preview_cols = ["From", "Subject", "body_raw", "body", "signature"]
# stripped = df_expanded["body_raw"].fillna("").ne(df_expanded["body"].fillna(""))
# preview = df_expanded.loc[stripped, preview_cols].head(3).copy()
# for col in ("body_raw", "body", "signature"):
#     preview[col] = preview[col].fillna("").str.slice(0, 400)
# preview

,From,Subject,body_raw,body,signature
3,phillip.allen@enron.com,,"Randy,\n\n Can you send me a schedule of the s...","Randy,\n\n Can you send me a schedule of the s...",Phillip
5,phillip.allen@enron.com,Re: Hello,"Greg,\n\n How about either next Tuesday or Thu...","Greg,\n\n How about either next Tuesday or Thu...",Phillip
6,phillip.allen@enron.com,,Please cc the following distribution list with...,Please cc the following distribution list with...,Phillip Allen (pallen@enron.com)\nMike Grigsby...


In [ ]:
# df = df_expanded.copy()

In [ ]:
# df = df.to_csv('emails_expanded.csv', index=False)

In [9]:
df = pd.read_csv('emails_expanded.csv')

/tmp/ipykernel_23994/4079247042.py:1: DtypeWarning: Columns (18: Time, 19: Attendees, 20: Re) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('emails_expanded.csv')


In [10]:
df.head()

,file,Message-ID,Date,From,To,Cc,Bcc,Subject,Mime-Version,Content-Type,...,Time,Attendees,Re,date_parsed,body_raw,body,signature,attachment_count,attachment_names,parse_error
0,allen-p/_sent_mail/1.,<18782981.1075855378110.JavaMail.evans@thyme>,"Mon, 14 May 2001 16:39:00 -0700",phillip.allen@enron.com,tim.belden@enron.com,NaN,NaN,NaN,1.0,"text/plain; charset=""us-ascii""",...,NaN,NaN,NaN,2001-05-14 16:39:00-07:00,Here is our forecast,Here is our forecast,NaN,0,[],NaN
1,allen-p/_sent_mail/10.,<15464986.1075855378456.JavaMail.evans@thyme>,"Fri, 04 May 2001 13:51:00 -0700",phillip.allen@enron.com,john.lavorato@enron.com,NaN,NaN,Re:,1.0,"text/plain; charset=""us-ascii""",...,NaN,NaN,NaN,2001-05-04 13:51:00-07:00,Traveling to have a business meeting takes the...,Traveling to have a business meeting takes the...,NaN,0,[],NaN
2,allen-p/_sent_mail/100.,<24216240.1075855687451.JavaMail.evans@thyme>,"Wed, 18 Oct 2000 03:00:00 -0700",phillip.allen@enron.com,leah.arsdall@enron.com,NaN,NaN,Re: test,1.0,"text/plain; charset=""us-ascii""",...,NaN,NaN,NaN,2000-10-18 03:00:00-07:00,test successful. way to go!!!,test successful. way to go!!!,NaN,0,[],NaN
3,allen-p/_sent_mail/1000.,<13505866.1075863688222.JavaMail.evans@thyme>,"Mon, 23 Oct 2000 06:13:00 -0700",phillip.allen@enron.com,randall.gay@enron.com,NaN,NaN,NaN,1.0,"text/plain; charset=""us-ascii""",...,NaN,NaN,NaN,2000-10-23 06:13:00-07:00,"Randy,\n\n Can you send me a schedule of the s...","Randy,\n\n Can you send me a schedule of the s...",Phillip,0,[],NaN
4,allen-p/_sent_mail/1001.,<30922949.1075863688243.JavaMail.evans@thyme>,"Thu, 31 Aug 2000 05:07:00 -0700",phillip.allen@enron.com,greg.piper@enron.com,NaN,NaN,Re: Hello,1.0,"text/plain; charset=""us-ascii""",...,NaN,NaN,NaN,2000-08-31 05:07:00-07:00,Let's shoot for Tuesday at 11:45.,Let's shoot for Tuesday at 11:45.,NaN,0,[],NaN


## EDA

In [11]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 517401 entries, 0 to 517400
Data columns (total 28 columns):
 #   Column                     Non-Null Count   Dtype  
---  ------                     --------------   -----  
 0   file                       517401 non-null  str    
 1   Message-ID                 517401 non-null  str    
 2   Date                       517401 non-null  str    
 3   From                       517401 non-null  str    
 4   To                         495554 non-null  str    
 5   Cc                         127881 non-null  str    
 6   Bcc                        127881 non-null  str    
 7   Subject                    498214 non-null  str    
 8   Mime-Version               517372 non-null  float64
 9   Content-Type               517372 non-null  str    
 10  Content-Transfer-Encoding  517372 non-null  str    
 11  X-From                     517372 non-null  str    
 12  X-To                       508248 non-null  str    
 13  X-cc                       128886 non-nu

In [14]:
df.columns

Index(['file', 'Message-ID', 'Date', 'From', 'To', 'Cc', 'Bcc', 'Subject',
       'Mime-Version', 'Content-Type', 'Content-Transfer-Encoding', 'X-From',
       'X-To', 'X-cc', 'X-bcc', 'X-Folder', 'X-Origin', 'X-FileName', 'Time',
       'Attendees', 'Re', 'date_parsed', 'body_raw', 'body', 'signature',
       'attachment_count', 'attachment_names', 'parse_error'],
      dtype='str')

In [13]:
df[df['To'].isna()].head()

,file,Message-ID,Date,From,To,Cc,Bcc,Subject,Mime-Version,Content-Type,...,Time,Attendees,Re,date_parsed,body_raw,body,signature,attachment_count,attachment_names,parse_error
188,allen-p/_sent_mail/264.,<15201149.1075855691021.JavaMail.evans@thyme>,"Mon, 01 May 2000 03:56:00 -0700",phillip.allen@enron.com,NaN,NaN,NaN,Re: DSL- Installs,1.0,"text/plain; charset=""us-ascii""",...,NaN,NaN,NaN,2000-05-01 03:56:00-07:00,No one will be home on 5/11/00 to meet DSL ins...,No one will be home on 5/11/00 to meet DSL ins...,"Call with questions. X37041.\n\nThank you,\n\n...",0,[],NaN
603,allen-p/all_documents/10.,<21975671.1075855665520.JavaMail.evans@thyme>,"Wed, 13 Dec 2000 08:35:00 -0800",messenger@ecm.bloomberg.com,NaN,NaN,NaN,Bloomberg Power Lines Report,1.0,"text/plain; charset=""ANSI_X3.4-1968""",...,NaN,NaN,NaN,2000-12-13 08:35:00-08:00,Here is today's copy of Bloomberg Power Lines....,Here is today's copy of Bloomberg Power Lines....,- daily.pdf,0,[],NaN
781,allen-p/all_documents/263.,<9828978.1075855671241.JavaMail.evans@thyme>,"Mon, 01 May 2000 03:56:00 -0700",phillip.allen@enron.com,NaN,NaN,NaN,Re: DSL- Installs,1.0,"text/plain; charset=""us-ascii""",...,NaN,NaN,NaN,2000-05-01 03:56:00-07:00,No one will be home on 5/11/00 to meet DSL ins...,No one will be home on 5/11/00 to meet DSL ins...,"Call with questions. X37041.\n\nThank you,\n\n...",0,[],NaN
873,allen-p/all_documents/348.,<8236042.1075855673105.JavaMail.evans@thyme>,"Fri, 07 Jan 2000 16:23:00 -0800",owner-strawbale@crest.org,NaN,NaN,NaN,NaN,1.0,"text/plain; charset=""us-ascii""",...,NaN,NaN,NaN,2000-01-07 16:23:00-08:00,<4DDE116DBCA1D3118B130080C840BAAD02CD53@ppims....,<4DDE116DBCA1D3118B130080C840BAAD02CD53@ppims....,NaN,0,[],NaN
885,allen-p/all_documents/359.,<26959382.1075855693279.JavaMail.evans@thyme>,"Mon, 14 May 2001 09:04:00 -0700",messenger@ecm.bloomberg.com,NaN,NaN,NaN,Bloomberg Power Lines Report,1.0,"text/plain; charset=""ANSI_X3.4-1968""",...,NaN,NaN,NaN,2001-05-14 09:04:00-07:00,Here is today's copy of Bloomberg Power Lines....,Here is today's copy of Bloomberg Power Lines....,- daily.pdf,0,[],NaN


In [15]:
df['Content-Type-no-charset'] = df['Content-Type'].str.split(';').str[0]
df['Content-Type-no-charset'].value_counts()

Content-Type-no-charset
text/plain    517372
Name: count, dtype: int64

In [16]:
df['attachment_count'].value_counts()

attachment_count
0    517401
Name: count, dtype: int64

In [18]:
# full_message using subject + body
df['full_message'] = df['Subject'] + ' ' + df['body']
df.duplicated(subset=['full_message']).sum()

np.int64(278270)

In [19]:
df = df.drop_duplicates(subset=['full_message'])
df.info()

<class 'pandas.DataFrame'>
Index: 239131 entries, 0 to 517400
Data columns (total 30 columns):
 #   Column                     Non-Null Count   Dtype  
---  ------                     --------------   -----  
 0   file                       239131 non-null  str    
 1   Message-ID                 239131 non-null  str    
 2   Date                       239131 non-null  str    
 3   From                       239131 non-null  str    
 4   To                         230770 non-null  str    
 5   Cc                         57801 non-null   str    
 6   Bcc                        57801 non-null   str    
 7   Subject                    239130 non-null  str    
 8   Mime-Version               239103 non-null  float64
 9   Content-Type               239103 non-null  str    
 10  Content-Transfer-Encoding  239103 non-null  str    
 11  X-From                     239103 non-null  str    
 12  X-To                       234174 non-null  str    
 13  X-cc                       58360 non-null   s